In [7]:
!pip install -q pyvi
!pip install -q pycocoevalcap
!pip install -q pyvi


In [8]:
import json
from collections import defaultdict

VINTERN_FILE = 'vintern_inference_results.json'
METADATA_FILE = 'metadata.jsonl'
QWEN_FILE = 'results_qwen2vl_test.jsonl'
OUTPUT_MERGED = 'comparison_results.json'

# ── 1. Đọc metadata → thứ tự file_name + captions ──
seen_order = []
file_to_captions = defaultdict(list)
with open(METADATA_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        fn = item['file_name']
        if fn not in file_to_captions:
            seen_order.append(fn)
        file_to_captions[fn].append(item['caption'])

# ── 2. Đọc Vintern → gán file_name theo index ──
with open(VINTERN_FILE, 'r', encoding='utf-8') as f:
    vintern_raw = json.load(f)
vintern_list = vintern_raw['predictions']

# Tạo dict: file_name → prediction Vintern
vintern_by_fn = {}
for i, item in enumerate(vintern_list):
    fn = seen_order[i]   # gán file_name theo thứ tự đã xác nhận khớp
    vintern_by_fn[fn] = item['prediction']

# ── 3. Đọc Qwen → tạo dict: file_name → prediction Qwen ──
qwen_by_fn = {}
with open(QWEN_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        qwen_by_fn[item['file_name']] = item['prediction']

# ── 4. Ghép dựa trên file_name ──
merged_data = []
missing_qwen = []

for i, fn in enumerate(seen_order):
    if fn not in qwen_by_fn:
        missing_qwen.append(fn)
        continue
    merged_data.append({
        "image_id": i,
        "file_name": fn,
        "references": file_to_captions[fn],      # 5 captions GT
        "vintern_pred": vintern_by_fn[fn],
        "qwen2vl_pred": qwen_by_fn[fn]
    })

# ── 5. Lưu kết quả ──
with open(OUTPUT_MERGED, 'w', encoding='utf-8') as f:
    json.dump(merged_data, f, ensure_ascii=False, indent=4)

# ── 6. Báo cáo ──
print(f"✅ Đã ghép xong: {len(merged_data)} ảnh")
print(f"📁 Lưu tại: {OUTPUT_MERGED}")
if missing_qwen:
    print(f"⚠️  {len(missing_qwen)} ảnh có trong Vintern/Metadata nhưng THIẾU trong Qwen:")
    for fn in missing_qwen[:10]:
        print(f"   - {fn}")
else:
    print("✅ Tất cả 800 ảnh đều có đủ trong cả 3 nguồn!")

# ── 7. Xem mẫu kiểm tra ──
print("\n📋 Mẫu record đầu tiên:")
print(json.dumps(merged_data[0], ensure_ascii=False, indent=2))


✅ Đã ghép xong: 800 ảnh
📁 Lưu tại: comparison_results.json
✅ Tất cả 800 ảnh đều có đủ trong cả 3 nguồn!

📋 Mẫu record đầu tiên:
{
  "image_id": 0,
  "file_name": "00009.jpg",
  "references": [
    "tuyến phố này đông đúc với nhiều xe máy đang lưu thông hai chiều và vỉa hè hai bên bị bủa vây bởi bàn ghế cùng phương tiện, bạn buộc phải đi bộ cẩn thận dưới mép đường nhựa sát lề",
    "khu vực đường sá nhộn nhịp có một chiếc taxi màu xanh đang chạy tới và phần lề phải hoàn toàn bị chắn bởi các biển hiệu, hãy đi sát mép lề và chú ý tiếng động cơ xe đang tới gần",
    "phía trước là đoạn đường hẹp có mật độ giao thông khá cao cùng nhiều cửa hàng kinh doanh lấn chiếm toàn bộ lối đi vỉa hè, bạn nên đi chậm sát lề phải và dùng gậy kiểm tra vật cản",
    "không gian hai bên đường tràn ngập các mái che và xe gắn máy dừng đỗ ngổn ngang khiến người đi bộ không còn khoảng trống an toàn, hãy cảnh giác cao độ và lắng nghe xe cộ đang di chuyển",
    "lòng đường đang có nhiều phương tiện hỗn hợp chạy ng

In [9]:
import json
import os
import pandas as pd
from pyvi import ViTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.rouge.rouge import Rouge

# --- 3. HÀM CHẤM ĐIỂM CHUẨN TIẾNG VIỆT ---
def calculate_metrics(predictions, model_name):
    print(f"📝 Đang chấm điểm cho {model_name}...")
    
    gts = {}
    res = {}
    
    for item in predictions:
        idx = str(item['image_id'])
        # Tách từ cho câu dự đoán của model này
        pred_text = item[model_name]
        pred_seg = ViTokenizer.tokenize(pred_text.lower())
        res[idx] = [pred_seg]
        
        # Tách từ cho 5 câu mẫu (References dùng chung)
        refs_seg = [ViTokenizer.tokenize(ref.lower()) for ref in item['references']]
        gts[idx] = refs_seg

    # Khởi tạo các hàm đo
    scorers = [
        (Bleu(4), ["BLEU-1", "BLEU-2", "BLEU-3", "BLEU-4"]),
        (Cider(), "CIDEr"),
        (Rouge(), "ROUGE-L")
    ]
    
    final_scores = {}
    for scorer, method in scorers:
        score, _ = scorer.compute_score(gts, res)
        if isinstance(method, list):
            for m, s in zip(method, score):
                final_scores[m] = round(s * 100, 2)
        else:
            final_scores[method] = round(score * 100, 2)
            
    return final_scores

# --- 4. THỰC THI CHẤM ĐIỂM ---
scores_vintern = calculate_metrics(merged_data, "vintern_pred")
scores_qwen = calculate_metrics(merged_data, "qwen2vl_pred")

# --- 5. TỔNG HỢP VÀ IN BẢNG SO SÁNH ---
# Thêm thông số hệ thống vào bảng (lấy từ file Vintern và Qwen bạn đã chạy)
# Chú ý: Params và Time bạn tự điền con số thực tế đo được vào đây
comparison_table = {
    "Vintern-1B-v3_5": {
        **scores_vintern,
        "Time/Img (s)": vintern_raw['system_metrics']['time_per_img_sec'],
        "VRAM (GB)": vintern_raw['system_metrics']['peak_vram_GB'],
        "Params (M)": vintern_raw['system_metrics']['params_M']
    },
    "Qwen2-VL-2B": {
        **scores_qwen,
        "Time/Img (s)": 2.8127, # Điền con số bạn đo được của Qwen vào đây
        "VRAM (GB)": 4.13,     # Điền con số bạn đo được của Qwen vào đây
        "Params (M)": 2210  # Điền con số bạn đo được của Qwen vào đây
    }
}

df = pd.DataFrame(comparison_table).T
print("\n" + "="*80)
print("🏆 BẢNG SO SÁNH KẾT QUẢ CUỐI CÙNG")
print("="*80)
print(df.to_string())
print("="*80)



📝 Đang chấm điểm cho vintern_pred...
{'testlen': 37590, 'reflen': 27628, 'guess': [37590, 36790, 35990, 35190], 'correct': [13716, 2496, 643, 200]}
ratio: 1.3605762270160213
📝 Đang chấm điểm cho qwen2vl_pred...
{'testlen': 31694, 'reflen': 26892, 'guess': [31694, 30894, 30094, 29294], 'correct': [12026, 2210, 425, 73]}
ratio: 1.1785661163170764

🏆 BẢNG SO SÁNH KẾT QUẢ CUỐI CÙNG
                 BLEU-1  BLEU-2  BLEU-3  BLEU-4  CIDEr  ROUGE-L  Time/Img (s)  VRAM (GB)  Params (M)
Vintern-1B-v3_5   36.49   15.73    7.62    3.98   3.89    17.66        4.4005       2.30      938.19
Qwen2-VL-2B       37.94   16.48    7.26    3.13   4.52    17.66        2.8127       4.13     2210.00
